In [1]:
# 09_cost_benefit_and_monte_carlo.ipynb
# Scenario-based actuarial cost-benefit analysis with a corrected, internally consistent
# loss-ratio calculation, plus a probabilistic (Monte Carlo) sensitivity analysis.
#
# Accounting conventions (all per 1,000 screened high-risk policyholders, 10-year horizon):
#   premium_base   = 1,000 * annual_premium * 10            (e.g., 1,000 * $1,200 * 10 = $12.0M)
#   prevented      = 1,000 * uptake(pi) * efficacy(eps)
#   averted_claims = prevented * dL           (dL = averted claims per prevented case, 10-yr)
#   screening_cost = 1,000 * c_screen
#   net_benefit    = averted_claims - screening_cost        (net of programme cost)
#   loss_ratio_pp  = 100 * averted_claims / premium_base    (claims are the loss-ratio numerator)
# NOTE: net_benefit and averted_claims are distinct quantities; only averted_claims enters the
# loss ratio. dL is defined over the 10-year horizon (NOT lifetime) so numerator and denominator
# share a horizon; an optional discounted variant (rate r) is also reported. Constant retention
# (no lapse) is assumed. The formula is CONDITIONAL on the elevated risk of the screened cohort;
# it does not by itself establish superiority of targeted over random screening. dpi 600, PNG+PDF.

import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
SEED = 42; rng = np.random.default_rng(SEED)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
TAB = ROOT / "results" / "tables"; FIG = ROOT / "results" / "figures"

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42,
                     "ps.fonttype": 42, "font.size": 11, "axes.edgecolor": "0.2",
                     "axes.linewidth": 0.8, "grid.color": "0.85"})
def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

N = 1000; ANNUAL_PREMIUM = 1200; HORIZON = 10
PREMIUM_BASE = N * ANNUAL_PREMIUM * HORIZON          # 12,000,000
DISC_RATE = 0.03

def npv_factor(r, years):
    # average discount factor applied to a level 10-year stream
    return np.mean([(1 + r) ** -(t + 1) for t in range(years)])

DF = npv_factor(DISC_RATE, HORIZON)

def scenario(pi, eps, dL, c_screen=50):
    prevented = N * pi * eps
    averted = prevented * dL
    screening = N * c_screen
    net = averted - screening
    lr_pp = 100 * averted / PREMIUM_BASE
    lr_pp_disc = 100 * (averted * DF) / (PREMIUM_BASE * DF)   # equal horizon -> same ratio
    return prevented, averted, screening, net, lr_pp

# ---- Deterministic scenarios ----
scen = {"Conservative": (0.10, 0.15, 8000), "Base": (0.15, 0.20, 10000),
        "Optimistic": (0.20, 0.25, 12000)}
rows = []
for name, (pi, eps, dL) in scen.items():
    prevented, averted, screening, net, lr = scenario(pi, eps, dL)
    rows.append([name, pi, eps, dL, round(prevented, 1), round(averted/1000, 1),
                 round(screening/1000, 1), round(net/1000, 1), round(lr, 2)])
cb = pd.DataFrame(rows, columns=["Scenario", "Uptake", "Efficacy", "dL_per_case",
                                 "Prevented_cases", "Averted_claims_$K", "Screening_cost_$K",
                                 "Net_benefit_$K", "LossRatio_improve_pp"])
cb.to_csv(TAB / "table_cost_benefit.csv", index=False)
print("Scenario cost-benefit (per 1,000 screened; premium base $12.0M over 10y):")
print(cb.to_string(index=False))

# ---- Monte Carlo sensitivity ----
M = 50000
def beta_on(lo, hi, mode_rel=0.5, conc=12):
    a = 1 + conc * mode_rel; b = 1 + conc * (1 - mode_rel)
    return lo + (hi - lo) * rng.beta(a, b, M)
pi = beta_on(0.10, 0.20); eps = beta_on(0.15, 0.30)
dL = rng.triangular(8000, 10000, 12000, M); c_screen = rng.uniform(30, 50, M)
prevented = N * pi * eps; averted = prevented * dL
net = averted - N * c_screen; lr = 100 * averted / PREMIUM_BASE

def ci(x): return np.percentile(x, [2.5, 50, 97.5])
net_ci = ci(net) / 1000; lr_ci = ci(lr)
mc = pd.DataFrame({
    "Metric": ["Net benefit ($K per 1,000)", "Loss-ratio improvement (pp)"],
    "P2.5": [round(net_ci[0], 1), round(lr_ci[0], 2)],
    "Median": [round(net_ci[1], 1), round(lr_ci[1], 2)],
    "P97.5": [round(net_ci[2], 1), round(lr_ci[2], 2)],
    "P(>0)": [round((net > 0).mean(), 3), round((lr > 0).mean(), 3)],
})
mc.to_csv(TAB / "table_monte_carlo.csv", index=False)
print("\nMonte Carlo (50,000 draws):")
print(mc.to_string(index=False))

# ---- Tornado: one-at-a-time swing to 5th/95th percentiles ----
base_pi, base_eps, base_dL, base_c = 0.15, 0.225, 10000, 40
def lr_val(pi_, eps_, dL_, c_):
    return 100 * (N * pi_ * eps_ * dL_) / PREMIUM_BASE
params = {"Uptake": (np.percentile(pi, 5), np.percentile(pi, 95), "pi"),
          "Efficacy": (np.percentile(eps, 5), np.percentile(eps, 95), "eps"),
          "Averted claims dL": (8000, 12000, "dL")}
tor = []
for name, (lo, hi, key) in params.items():
    kw_lo = dict(pi_=base_pi, eps_=base_eps, dL_=base_dL, c_=base_c)
    kw_hi = dict(kw_lo)
    kw_lo[{"pi": "pi_", "eps": "eps_", "dL": "dL_"}[key]] = lo
    kw_hi[{"pi": "pi_", "eps": "eps_", "dL": "dL_"}[key]] = hi
    tor.append([name, round(lr_val(**kw_lo), 2), round(lr_val(**kw_hi), 2),
                round(abs(lr_val(**kw_hi) - lr_val(**kw_lo)), 2)])
tor = pd.DataFrame(tor, columns=["Parameter", "Low_pp", "High_pp", "Swing_pp"]).sort_values("Swing_pp")
tor.to_csv(TAB / "table_tornado.csv", index=False)

# ---- Figures ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, data, lab, c in [(axes[0], net/1000, "Net benefit ($K per 1,000 screened)", "#6f6f6f"),
                         (axes[1], lr, "Loss-ratio improvement (pp)", "#6f6f6f")]:
    ax.hist(data, bins=50, color=c, edgecolor="black", linewidth=0.3)
    q = np.percentile(data, [2.5, 50, 97.5])
    ax.axvline(q[1], color="black", lw=1.5, label="Median")
    ax.axvline(q[0], color="black", ls="--", lw=1); ax.axvline(q[2], color="black", ls="--", lw=1, label="95% interval")
    ax.set_xlabel(lab); ax.set_ylabel("Frequency"); ax.legend(frameon=False)
axes[0].text(0.04, 0.92, "(A)", transform=axes[0].transAxes, fontsize=10, fontweight="bold")
axes[1].text(0.04, 0.92, "(B)", transform=axes[1].transAxes, fontsize=10, fontweight="bold")
savefig(fig, "fig_monte_carlo")

fig, ax = plt.subplots(figsize=(7, 3.2))
yy = np.arange(len(tor)); base_lr = lr_val(base_pi, base_eps, base_dL, base_c)
for i, (_, r) in enumerate(tor.iterrows()):
    ax.barh(i, r["High_pp"] - r["Low_pp"], left=r["Low_pp"], color="#9a9a9a", edgecolor="black", linewidth=0.6)
ax.axvline(base_lr, color="black", ls="--", lw=1, label="Base case")
ax.set_yticks(yy); ax.set_yticklabels(tor["Parameter"]); ax.set_xlabel("Loss-ratio improvement (pp)")
ax.legend(frameon=False); savefig(fig, "fig_tornado")

json.dump({"premium_base": PREMIUM_BASE, "horizon": HORIZON, "annual_premium": ANNUAL_PREMIUM,
           "net_benefit_median_$K": float(net_ci[1]), "lossratio_median_pp": float(lr_ci[1]),
           "P_netbenefit_positive": float((net > 0).mean())},
          open(TAB / "cost_benefit_metadata.json", "w"), indent=2)
print("\nTornado (loss-ratio pp swing):")
print(tor.to_string(index=False))
print("\nSaved cost-benefit, Monte Carlo, tornado tables and figures.")


Scenario cost-benefit (per 1,000 screened; premium base $12.0M over 10y):
    Scenario  Uptake  Efficacy  dL_per_case  Prevented_cases  Averted_claims_$K  Screening_cost_$K  Net_benefit_$K  LossRatio_improve_pp
Conservative    0.10      0.15         8000             15.0              120.0               50.0            70.0                   1.0
        Base    0.15      0.20        10000             30.0              300.0               50.0           250.0                   2.5
  Optimistic    0.20      0.25        12000             50.0              600.0               50.0           550.0                   5.0

Monte Carlo (50,000 draws):
                     Metric   P2.5  Median  P97.5  P(>0)
 Net benefit ($K per 1,000) 207.70  294.70 402.60    1.0
Loss-ratio improvement (pp)   2.07    2.79   3.68    1.0



Tornado (loss-ratio pp swing):
        Parameter  Low_pp  High_pp  Swing_pp
           Uptake    2.41     3.21      0.80
         Efficacy    2.41     3.21      0.80
Averted claims dL    2.25     3.38      1.12

Saved cost-benefit, Monte Carlo, tornado tables and figures.
